# Manufacturing LLM Fine-Tuning with LoRA and QLoRA

This notebook fine-tunes an open-source model on synthetic manufacturing maintenance and quality data.

**Groq is used only for optional hosted inference or LLM-as-a-Judge evaluation.**
LoRA/QLoRA training runs locally or in Google Colab using Hugging Face, PEFT, TRL, PyTorch, and bitsandbytes.

Recommended runtime: **Google Colab GPU (T4/L4)**.

In [ ]:
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes sentencepiece groq python-dotenv

In [ ]:
import json, os, random
from pathlib import Path
import torch
from datasets import Dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, set_seed
from trl import SFTTrainer

SEED = 42
set_seed(SEED)
BASE_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
TRAINING_MODE = "qlora"  # change to "lora" if needed
OUTPUT_DIR = "./outputs/manufacturing-adapter"

In [ ]:
data_file = Path("data/manufacturing_instructions.jsonl")
if not data_file.exists():
    data_file = Path("/content/manufacturing_instructions.jsonl")
if not data_file.exists():
    raise FileNotFoundError("Upload manufacturing_instructions.jsonl or run from the extracted project folder.")

records = [json.loads(line) for line in data_file.read_text(encoding="utf-8").splitlines() if line.strip()]
random.shuffle(records)
split = max(1, int(0.8 * len(records)))
train_ds = Dataset.from_list(records[:split])
eval_ds = Dataset.from_list(records[split:])
print(len(train_ds), len(eval_ds))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

SYSTEM = "You are a manufacturing maintenance and quality engineering assistant. Give safety-aware, technically grounded answers."

def format_example(example):
    messages = [
        {"role":"system","content":SYSTEM},
        {"role":"user","content":f"Instruction: {example['instruction']}\nCase: {example['input']}"},
        {"role":"assistant","content":example["output"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

In [ ]:
dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16

quant_config = None
if TRAINING_MODE == "qlora":
    if not torch.cuda.is_available():
        raise RuntimeError("QLoRA requires a CUDA GPU in this notebook.")
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=dtype,
    )

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=None if quant_config else (dtype if torch.cuda.is_available() else torch.float32),
    device_map="auto" if torch.cuda.is_available() else None,
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
def build_prompt(case):
    messages = [
        {"role":"system","content":SYSTEM},
        {"role":"user","content":f"Diagnose the issue, list likely causes, checks, and actions.\nCase: {case}"}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate(active_model, case, max_new_tokens=160):
    prompt = build_prompt(case)
    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(active_model.parameters()).device
    inputs = {k:v.to(device) for k,v in inputs.items()}
    out = active_model.generate(
        **inputs, max_new_tokens=max_new_tokens, do_sample=False,
        repetition_penalty=1.05, pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

TEST_CASE = "A stamping press shows rising hydraulic-oil temperature, slow ram return, and intermittent low-pressure alarms."
baseline = generate(model, TEST_CASE)
print(baseline)

In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    report_to="none",
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=peft_config,
    formatting_func=format_example,
)

trainer.model.print_trainable_parameters()
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

In [ ]:
tuned = generate(trainer.model, TEST_CASE)
print("BASE MODEL\n", baseline)
print("\n" + "="*80 + "\n")
print("FINE-TUNED MODEL\n", tuned)

## Optional Groq comparison

Groq currently hosts inference models such as `llama-3.1-8b-instant`. Confirm the active model ID in your Groq console before running.

In [ ]:
# Optional: set GROQ_API_KEY and uncomment.
#
# from groq import Groq
# client = Groq(api_key=os.environ["GROQ_API_KEY"])
# response = client.chat.completions.create(
#     model="llama-3.1-8b-instant",
#     temperature=0,
#     messages=[
#         {"role":"system","content":SYSTEM},
#         {"role":"user","content":f"Diagnose the manufacturing issue.\nCase: {TEST_CASE}"}
#     ],
# )
# print(response.choices[0].message.content)

## Portfolio next steps

- Expand to 500–2,000 expert-reviewed examples.
- Keep a separate expert-labelled test set.
- Track experiments with MLflow or Weights & Biases.
- Evaluate diagnostic completeness, safety, factuality, and hallucination.
- Compare LoRA vs QLoRA memory usage, training time, and quality.
- Add FastAPI inference and a simple Streamlit demo.